In [1]:
stroka = 'tyt,ytyl,kl'
stroka = stroka.strip(',')
print(stroka)

tyt,ytyl,kl


In [108]:
import pandas as pd
import sqlite3
df = pd.read_csv(r'C:\Users\Gleb\Desktop\Курсы\Стажировки\flowwow\orders.csv')

display(df.sort_values(by = 'user_id'))

,order_id,user_id,order_date,product_id,category,price,quantity,region
26,10005,1,2024-01-25,A002,gadgets,2990,2,kazan
28,10002,1,2024-01-26,A001,gadgets,990,1,novosibirsk
2,10019,2,2024-01-03,A002,gadgets,2490,3,kazan
11,10028,2,2024-01-14,D001,home,2490,2,kazan
19,10029,2,2024-01-20,B001,books,1990,1,ekaterinburg
17,10015,2,2024-01-19,C001,sports,2990,1,novosibirsk
20,10007,2,2024-01-20,B002,books,490,2,kazan
3,10025,3,2024-01-06,D001,home,990,3,ekaterinburg
27,10021,3,2024-01-25,B001,books,990,1,novosibirsk
25,10016,4,2024-01-23,A002,gadgets,1990,2,kazan


In [17]:
conn = sqlite3.connect(":memory:")
df.to_sql("orders", conn, index=False, if_exists="replace")

30

# SQL

**1 Задача**

In [98]:
dau = """ 
SELECT order_date as date, COUNT(DISTINCT user_id) as DAU FROM orders
GROUP BY order_date
ORDER BY date	
""" 

In [99]:
result1 = pd.read_sql(dau, conn).set_index('date')
display(result1)

,DAU
date,
2024-01-01,1
2024-01-02,1
2024-01-03,1
2024-01-06,1
2024-01-07,2
2024-01-08,1
2024-01-11,1
2024-01-13,1
2024-01-14,2


In [100]:
mau = """ 
SELECT strftime('%Y-%m', order_date) as month, COUNT(DISTINCT user_id) as MAU FROM orders 
GROUP BY 1
ORDER BY month
""" 

# date_format & date_trunc не будут работать

In [101]:
result2 = pd.read_sql(mau, conn).set_index('month')
display(result2)

,MAU
month,
2024-01,10


In [102]:
avg_dau = """
WITH dau as (
SELECT order_date as date, COUNT(DISTINCT user_id) as DAU FROM orders
GROUP BY order_date
ORDER BY date	
)

SELECT strftime('%Y-%m', date) as month, ROUND(AVG(DAU), 2) as AVG_DAU FROM dau
GROUP BY 1
ORDER BY month
"""

In [103]:
result3 = pd.read_sql(avg_dau, conn).set_index('month')
display(result3)

,AVG_DAU
month,
2024-01,1.42


`Здесь CR формально будет = 1, потому что в SQLite-таблице с заказами нет пользователей без оплаты`

In [104]:
CR = """
SELECT
    strftime('%Y-%m', order_date) AS month,
    COUNT(DISTINCT user_id) AS paying_users,
    COUNT(DISTINCT user_id) * 1.0
        / COUNT(DISTINCT user_id) AS CR
FROM orders
GROUP BY month
ORDER BY month
"""

In [105]:
result4 = pd.read_sql(CR, conn).set_index('month')
display(result4)

,paying_users,CR
month,,
2024-01,10,1.0


**2 задача на дубли**

In [106]:
print(df.duplicated().sum())

0


In [113]:
dublicats = """
WITH kolvo_dubl as (
SELECT *, COUNT(*) AS cnt
FROM orders
GROUP BY order_id, user_id, order_date,	product_id,	category, price, quantity,	region
HAVING COUNT(*) > 1
)

SELECT COUNT(*) as kovlo_dublikatov FROM kolvo_dubl
"""

In [116]:
dubl = pd.read_sql(dublicats, conn).set_index('kovlo_dublikatov')
display(dubl)

""
kovlo_dublikatov
0


**3 задача**

In [118]:
plat_avg = """
WITH user_payments as
(
SELECT user_id, SUM(price * quantity) AS total_payment
FROM orders
GROUP BY user_id
)
SELECT user_id, total_payment
FROM user_payments
WHERE total_payment > (SELECT AVG(total_payment) FROM user_payments)
"""

In [121]:
plat = pd.read_sql(plat_avg, conn).set_index('user_id')
display(plat)

,total_payment
user_id,
2,18410
4,11430
5,14920
9,22410


**4 задача**

`Для каждой календарной даты код выбираем всех учеников, которые были активны за последние 30 дней (включая текущую дату).                                                       Считаем кол-во уникальных учеников. Используем скользящее окно в 30дн.`

In [125]:
recurs = """
WITH RECURSIVE dates(date) AS (
    SELECT DATE('now', '-29 day')
    UNION ALL
    SELECT DATE(date, '+1 day')
    FROM dates
    WHERE date < DATE('now')
)
SELECT
    d.date,
    COUNT(DISTINCT s.student_id) AS rolling_30d_students
FROM dates as d
LEFT JOIN students as s ON DATE(s.event_date) >= DATE(d.date, '-29 day') AND DATE(s.event_date) <= d.date
GROUP BY d.date
ORDER BY d.date
"""
# DATE(d.date, '-29 day')  для SQLlite, для Postgre использовал бы DATE_ADD
# DATE(s.event_date) >= DATE(d.date, '-29 day') AND DATE(s.event_date) <= d.date Можно и через BETWEEN
# Для каждой даты берутся все события учеников, которые произошли за последние 30 дней до этой даты (включая её)

# Python

In [160]:
zadacha = """
SELECT user_id, COUNT(order_id) as cnt FROM orders
GROUP BY 1
ORDER BY cnt desc
"""
pl = pd.read_sql(zadacha, conn)
display(pl)


,user_id,cnt
0,9,5
1,5,5
2,2,5
3,10,4
4,4,4
5,3,2
6,1,2
7,8,1
8,7,1
9,6,1


**1 задача**

In [133]:
kolvo = df['order_id'].count()
print(kolvo)

30


**2 задача**

In [134]:
df['user_id'].nunique()

10

**3 задача**

In [140]:
df['gmv'] = df['price'] * df['quantity']
total_gmv = df['gmv'].sum()
print(total_gmv)

100940


**4 задача**

**5 задача**

In [159]:
orders_user_count = df[['user_id', 'order_id']].groupby('user_id').count()

one_order = (orders_user_count['order_id'] == 1).sum()
more_than_three = (orders_user_count['order_id'] > 3).sum()

print(f'''Кол-во пользователей с одним заказом: {one_order}
Кол-во пользователей с более, чем тремя заказами: {more_than_three}''')

Кол-во пользователей с одним заказом: 3
Кол-во пользователей с более, чем тремя заказами: 5


**6 задача**

In [173]:
user_gmv = df[['user_id', 'gmv']].groupby('user_id').sum()
top_user = user_gmv.loc[[user_gmv['gmv'].idxmax()], :]
display(top_user)


,gmv
user_id,
9,22410
